# Validation logic / reasonableness check

Sanity-checks the **two draft source files** before (or after) they're merged by
[`merge_validation.ipynb`](../../scripts/merge_validation.ipynb) -- it does not
require a merge to have happened yet, since it loads the cardinality/marginality
source and the JSON-Schema-rules source directly and compares them.

This is a QA pass, not a fix -- it produces a markdown report flagging things a
curator should look at, and doesn't change either input file.

**Run this notebook's kernel under the `bioschemas` conda environment**
(`Anaconda3\envs\bioschemas`). Only the standard library is used (`json`, `pathlib`,
`datetime`).

## 1. Parameters

In [ ]:
import json
from pathlib import Path
from datetime import date

ROOT_DIR = Path(r"C:\Users\gtsueng\Anaconda3\envs\bioschemas\schema_validation_merger")
INPUT_DIR = ROOT_DIR / "files" / "to merge"
OUTPUT_DIR = ROOT_DIR / "output"

# Hints only -- roles are cross-checked against actual content (see below), so this
# keeps working even if a file gets renamed, as happened mid-project already.
cardinality_source_file = "Nick-Sample-Profile.json"
schema_rules_source_file = "Ginger-Sample-Profile.json"

## 2. Load input files and detect roles

Same role-detection approach as the merger notebook: match by filename first, but
cross-check against which file actually has populated top-level
`required`/`recommended`/`optional` lists, since that's what really distinguishes
the cardinality/marginality source from the JSON-Schema-rules source.

In [ ]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def first_graph_validation(doc):
    for entry in doc.get("@graph", []):
        if "$validation" in entry:
            return entry, entry["$validation"]
    raise ValueError("No @graph entry with a $validation block was found.")


def has_marginality(validation):
    return any(validation.get(key) for key in ("required", "recommended", "optional"))


def normalize(name: str) -> str:
    return name.lower().replace("_", "-")


input_files = sorted(INPUT_DIR.glob("*.json"))
if not input_files:
    raise FileNotFoundError(f"No JSON files found in {INPUT_DIR}")

by_name = {normalize(f.name): f for f in input_files}
cardinality_path = by_name.get(normalize(cardinality_source_file))
schema_rules_path = by_name.get(normalize(schema_rules_source_file))

if cardinality_path is None or schema_rules_path is None or cardinality_path == schema_rules_path:
    print("Configured filenames not both found -- auto-detecting roles from content instead.")
    candidates = []
    for f in input_files:
        doc = load_json(f)
        _, validation = first_graph_validation(doc)
        candidates.append((f, doc, has_marginality(validation)))
    marginality_files = [f for f, _, has_marg in candidates if has_marg]
    if len(marginality_files) == 1:
        cardinality_path = marginality_files[0]
        schema_rules_path = next(f for f, _, has_marg in candidates if not has_marg)
    else:
        raise ValueError(
            "Could not auto-detect roles -- more than one (or zero) input files have "
            "populated required/recommended/optional lists. Set cardinality_source_file "
            "and schema_rules_source_file explicitly above."
        )

cardinality_doc = load_json(cardinality_path)
schema_rules_doc = load_json(schema_rules_path)

cardinality_entry, cardinality_validation = first_graph_validation(cardinality_doc)
schema_entry, schema_validation = first_graph_validation(schema_rules_doc)

print(f"Cardinality / marginality source: {cardinality_path.name}")
print(f"JSON-Schema rules source:         {schema_rules_path.name}")

## 3. Extract per-property info from each source

Same extraction as the merger: `cardinality_props` keeps only `description` /
`owl:cardinality`; `schema_props` keeps everything *except* that file's own
`description` (i.e. the actual JSON-Schema keywords).

In [ ]:
def extract_cardinality_info(validation):
    props = {}
    for name, body in validation.get("properties", {}).items():
        info = {}
        if "description" in body:
            info["description"] = body["description"]
        if "owl:cardinality" in body:
            info["owl:cardinality"] = body["owl:cardinality"]
        props[name] = info
    return props


def extract_schema_rules(validation):
    props = {}
    for name, body in validation.get("properties", {}).items():
        props[name] = {k: v for k, v in body.items() if k != "description"}
    return props


cardinality_props = extract_cardinality_info(cardinality_validation)
schema_props = extract_schema_rules(schema_validation)
all_property_names = sorted(set(cardinality_props) | set(schema_props))

print(f"{len(cardinality_props)} properties in cardinality source, "
      f"{len(schema_props)} properties in schema-rules source, "
      f"{len(all_property_names)} total (union).")

## 4. Check 1 -- properties present in only one source

For each property missing from one side, records which kind of information is
missing: cardinality/marginality info (description + owl:cardinality), or
JSON-Schema validation rules.

In [ ]:
one_sided_properties = []
for name in all_property_names:
    in_cardinality = name in cardinality_props
    in_schema = name in schema_props
    if in_cardinality and not in_schema:
        one_sided_properties.append({
            "property": name,
            "present_in": cardinality_path.name,
            "missing_from": schema_rules_path.name,
            "missing_info": "JSON-Schema validation rules",
        })
    elif in_schema and not in_cardinality:
        one_sided_properties.append({
            "property": name,
            "present_in": schema_rules_path.name,
            "missing_from": cardinality_path.name,
            "missing_info": "description / owl:cardinality / marginality",
        })

print(f"{len(one_sided_properties)} propert{'y is' if len(one_sided_properties)==1 else 'ies are'} one-sided.")
for row in one_sided_properties:
    print(f"  - {row['property']!r}: only in {row['present_in']} "
          f"(missing {row['missing_info']})")

## 5. Check 2 -- properties with no real JSON-Schema validation rules

A property can technically "exist" in the schema-rules source and still carry no
actual validation keywords -- e.g. an entry that only ever had a `description` and
nothing else. Check 1 wouldn't catch this (the property isn't *missing*, it's just
empty once its `description` is set aside), so it's checked separately here.

In [ ]:
empty_schema_rule_properties = [
    name for name in sorted(schema_props)
    if not schema_props[name]
]

print(f"{len(empty_schema_rule_properties)} propert"
      f"{'y' if len(empty_schema_rule_properties)==1 else 'ies'} "
      f"present in {schema_rules_path.name} with no JSON-Schema validation keywords:")
for name in empty_schema_rule_properties:
    print("  -", name)

## 6. Check 3 -- `owl:cardinality: many` should use `oneOf` / `anyOf`

If a property's `owl:cardinality` is `many` (i.e. it can take multiple values), its
JSON-Schema rules should almost always be written as a `oneOf`/`anyOf` between a
single value and an array of that value -- that's the standard bioschemas pattern
for repeatable properties. Only checked for properties present on both sides, since
checking it requires both the cardinality value and the schema rules.

In [ ]:
cardinality_schema_mismatches = []
for name in sorted(set(cardinality_props) & set(schema_props)):
    cardinality_value = cardinality_props[name].get("owl:cardinality", "")
    if str(cardinality_value).strip().lower() != "many":
        continue
    rules = schema_props[name]
    has_oneof = "oneOf" in rules
    has_anyof = "anyOf" in rules
    if not has_oneof and not has_anyof:
        cardinality_schema_mismatches.append({
            "property": name,
            "owl:cardinality": cardinality_value,
            "top_level_keywords": sorted(rules.keys()),
        })

print(f"{len(cardinality_schema_mismatches)} propert"
      f"{'y' if len(cardinality_schema_mismatches)==1 else 'ies'} "
      f"marked owl:cardinality=many without oneOf/anyOf:")
for row in cardinality_schema_mismatches:
    print(f"  - {row['property']!r} (keywords present: {row['top_level_keywords']})")

## 7. Build and write the markdown report

Named after the class being checked (its `rdfs:label`, falling back to the local
part of its `@id`) and today's date.

In [ ]:
class_name = (
    cardinality_entry.get("rdfs:label")
    or schema_entry.get("rdfs:label")
    or cardinality_entry.get("@id", "UnknownClass").split(":")[-1]
)
safe_class_name = "".join(c if c.isalnum() or c in "-_" else "_" for c in class_name)
today = date.today().isoformat()
report_filename = f"{safe_class_name}_validation_check_{today}.md"

In [ ]:
lines = []
lines.append(f"# Validation logic check: {class_name}")
lines.append("")
lines.append(f"- **Date:** {today}")
lines.append(f"- **Cardinality / marginality source:** `{cardinality_path.name}`")
lines.append(f"- **JSON-Schema rules source:** `{schema_rules_path.name}`")
lines.append(f"- **Total properties checked (union):** {len(all_property_names)}")
lines.append("")
lines.append(
    "This is a draft reasonableness check to help a curator spot likely problems "
    "before or after merging -- it does not change either source file, and a flag "
    "here is a prompt for human review, not necessarily an error."
)
lines.append("")

lines.append("## 1. Properties present in only one source")
lines.append("")
if one_sided_properties:
    lines.append("| Property | Present in | Missing from | Missing info |")
    lines.append("|---|---|---|---|")
    for row in one_sided_properties:
        lines.append(
            f"| `{row['property']}` | `{row['present_in']}` | "
            f"`{row['missing_from']}` | {row['missing_info']} |"
        )
else:
    lines.append("None -- every property appears in both sources.")
lines.append("")

lines.append("## 2. Properties with no real JSON-Schema validation rules")
lines.append("")
lines.append(
    f"Present in `{schema_rules_path.name}` but carrying no validation keywords "
    "besides (possibly) a description."
)
lines.append("")
if empty_schema_rule_properties:
    lines.append("| Property |")
    lines.append("|---|")
    for name in empty_schema_rule_properties:
        lines.append(f"| `{name}` |")
else:
    lines.append("None -- every property in the schema-rules source has real validation keywords.")
lines.append("")

lines.append("## 3. `owl:cardinality: many` without `oneOf` / `anyOf`")
lines.append("")
lines.append(
    "Properties marked as repeatable (`owl:cardinality: many`) whose JSON-Schema "
    "rules don't offer a single-value/array choice via `oneOf` or `anyOf` -- worth "
    "double-checking these can actually accept multiple values as written."
)
lines.append("")
if cardinality_schema_mismatches:
    lines.append("| Property | owl:cardinality | Top-level JSON-Schema keywords present |")
    lines.append("|---|---|---|")
    for row in cardinality_schema_mismatches:
        keywords = ", ".join(row["top_level_keywords"]) or "(none)"
        lines.append(f"| `{row['property']}` | {row['owl:cardinality']} | {keywords} |")
else:
    lines.append("None -- every `many` property uses `oneOf` or `anyOf`.")
lines.append("")

report_text = "\n".join(lines) + "\n"
print(report_text)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / report_filename
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Wrote report to: {report_path}")